# Last Mile Logistics Auditor
## Data Engineering Notebook

This notebook analyzes last-mile delivery performance using the Olist Brazilian E-Commerce dataset.

**Goal:** Find which regions and product categories have delivery problems, and how those problems affect customer satisfaction.

---
## Section 1: Import Libraries

We need pandas for data manipulation, numpy for calculations, matplotlib and plotly for charts.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import plotly.express as px
import plotly.graph_objects as go
import warnings

# Hide warnings to keep output clean
warnings.filterwarnings('ignore')

print('All libraries imported successfully')

---
## Section 2: Load Dataset

We load 6 CSV files from the `data/` folder. Each file contains different information:
- **orders** - order dates, delivery dates, status
- **reviews** - customer review scores
- **customers** - customer location (city, state)
- **order_items** - products in each order
- **products** - product details and category
- **translations** - Portuguese category names to English

In [ ]:
# Load all CSV files using relative paths
# The ../ means go one folder up from notebooks/ to reach data/

orders = pd.read_csv('../data/olist_orders_dataset.csv')
reviews = pd.read_csv('../data/olist_order_reviews_dataset.csv')
customers = pd.read_csv('../data/olist_customers_dataset.csv')
order_items = pd.read_csv('../data/olist_order_items_dataset.csv')
products = pd.read_csv('../data/olist_products_dataset.csv')
translations = pd.read_csv('../data/product_category_name_translation.csv')

print('All 6 datasets loaded successfully')

---
## Section 3: Explore Data

Before joining tables, we need to understand each dataset:
- How many rows and columns?
- What columns exist?
- Are there missing values?
- Are there duplicate rows?

In [ ]:
# Create a dictionary of all dataframes to explore them together
datasets = {
    'orders': orders,
    'reviews': reviews,
    'customers': customers,
    'order_items': order_items,
    'products': products,
    'translations': translations
}

# Show shape (rows, columns) for each dataset
print('=== DATASET SHAPES ===')
for name, df in datasets.items():
    print(f'{name:15} -> {df.shape[0]:,} rows, {df.shape[1]} columns')

In [ ]:
# Show column names for each dataset
print('=== COLUMN NAMES ===\n')
for name, df in datasets.items():
    print(f'{name}:')
    print(f'  {list(df.columns)}\n')

In [ ]:
# Check missing values for each dataset
print('=== MISSING VALUES ===\n')
for name, df in datasets.items():
    missing = df.isnull().sum()
    if missing.sum() > 0:
        print(f'{name}:')
        for col in missing[missing > 0].index:
            pct = missing[col] / len(df) * 100
            print(f'  {col}: {missing[col]:,} ({pct:.1f}%)')
        print()
    else:
        print(f'{name}: No missing values\n')

In [ ]:
# Check duplicate rows
print('=== DUPLICATE ROWS ===\n')
for name, df in datasets.items():
    dupes = df.duplicated().sum()
    print(f'{name:15} -> {dupes:,} duplicates')

---
## Section 4: Build Master Dataset

We join all tables into one master dataframe called `master_df`.

**Join order:**
1. orders + reviews (on `order_id`)
2. + customers (on `customer_id`)
3. + order_items (on `order_id`)
4. + products (on `product_id`)
5. + translations (on `product_category_name`)

**Important:** One order can have multiple items. After joining, we keep one record per `order_id` for delivery analysis.

In [ ]:
# Step 1: Join orders with reviews on order_id
# We merge on order_id to get the review score for each order

master_df = pd.merge(orders, reviews, on='order_id', how='left')
print(f'After orders + reviews: {master_df.shape[0]:,} rows')

# Step 2: Join with customers on customer_id
# This adds customer location info (city, state)

master_df = pd.merge(master_df, customers, on='customer_id', how='left')
print(f'After + customers: {master_df.shape[0]:,} rows')

# Step 3: Join with order_items on order_id
# This adds product information for each order

master_df = pd.merge(master_df, order_items, on='order_id', how='left')
print(f'After + order_items: {master_df.shape[0]:,} rows')

# Step 4: Join with products on product_id
# This adds product category names

master_df = pd.merge(master_df, products, on='product_id', how='left')
print(f'After + products: {master_df.shape[0]:,} rows')

# Step 5: Join with translations on product_category_name
# This converts Portuguese category names to English

master_df = pd.merge(master_df, translations, on='product_category_name', how='left')
print(f'After + translations: {master_df.shape[0]:,} rows')

In [ ]:
# IMPORTANT: One order can have multiple items
# For delivery analysis, we need one record per order_id

# Check how many items per order on average
items_per_order = master_df.groupby('order_id').size()
print(f'Unique orders: {items_per_order.shape[0]:,}')
print(f'Average items per order: {items_per_order.mean():.2f}')
print(f'Max items in one order: {items_per_order.max()}')

In [ ]:
# Keep one record per order_id for delivery analysis
# We take the first item's product info since delivery depends on the order, not individual items

master_df = master_df.drop_duplicates(subset='order_id', keep='first')
print(f'Master dataset: {master_df.shape[0]:,} rows (one per order)')
print(f'Columns: {master_df.shape[1]}')

---
## Section 5: Data Cleaning

We need to:
1. Convert date columns to datetime format
2. Remove canceled and unavailable orders
3. Remove orders without a delivery date

In [ ]:
# Convert date columns to datetime format
# This lets us do date calculations later

date_columns = [
    'order_purchase_timestamp',
    'order_delivered_customer_date',
    'order_estimated_delivery_date'
]

for col in date_columns:
    master_df[col] = pd.to_datetime(master_df[col])

print('Date columns converted to datetime format')
print(f'\nSample dates:')
print(master_df[['order_purchase_timestamp', 'order_delivered_customer_date', 'order_estimated_delivery_date']].head())

In [ ]:
# Check current order statuses before filtering
print('=== ORDER STATUS COUNTS ===')
print(master_df['order_status'].value_counts())

In [ ]:
# Remove canceled and unavailable orders
# These orders were not delivered, so we cannot measure delivery performance

rows_before = len(master_df)
master_df = master_df[~master_df['order_status'].isin(['canceled', 'unavailable'])]
print(f'Removed {rows_before - len(master_df):,} canceled/unavailable orders')

# Remove orders without delivery date
# If there is no delivery date, we cannot calculate delay

rows_before = len(master_df)
master_df = master_df.dropna(subset=['order_delivered_customer_date'])
print(f'Removed {rows_before - len(master_df):,} orders without delivery date')

print(f'\nFinal dataset: {len(master_df):,} orders')

---
## Section 6: Delivery Delay Calculation

We create three new columns:

1. **delay_days** = actual delivery - estimated delivery
   - Positive = late delivery
   - Negative or zero = on time

2. **days_difference** = estimated delivery - actual delivery
   - Positive = delivered early
   - Negative = delivered late

3. **delivery_status** = categorize delivery performance:
   - delay_days <= 0: On Time
   - delay_days 1-5: Late
   - delay_days > 5: Super Late

In [ ]:
# Calculate delay in days
# Positive value = order arrived AFTER estimated date (late)
# Negative value = order arrived BEFORE estimated date (early)

master_df['delay_days'] = (
    master_df['order_delivered_customer_date'] - master_df['order_estimated_delivery_date']
).dt.days

# days_difference is the reverse: how many days ahead of schedule
master_df['days_difference'] = (
    master_df['order_estimated_delivery_date'] - master_df['order_delivered_customer_date']
).dt.days

print('Delay columns created')
print(f'\nSample delay values:')
print(master_df[['order_id', 'delay_days', 'days_difference']].head(10))

In [ ]:
# Create delivery status categories
# This makes it easy to group and analyze delivery performance

def categorize_delivery(delay):
    if delay <= 0:
        return 'On Time'
    elif delay <= 5:
        return 'Late'
    else:
        return 'Super Late'

master_df['delivery_status'] = master_df['delay_days'].apply(categorize_delivery)

# Show the distribution of delivery statuses
print('=== DELIVERY STATUS DISTRIBUTION ===')
print(master_df['delivery_status'].value_counts())
print(f'\nPercentage:')
print(master_df['delivery_status'].value_counts(normalize=True).mul(100).round(1).astype(str) + '%')

In [ ]:
# Summary statistics for delay_days
print('=== DELAY STATISTICS ===')
print(f'Average delay: {master_df["delay_days"].mean():.1f} days')
print(f'Median delay: {master_df["delay_days"].median():.1f} days')
print(f'Max delay: {master_df["delay_days"].max()} days')
print(f'Min delay: {master_df["delay_days"].min()} days')

---
## Section 7: Geographic Analysis

We analyze delivery performance by customer state (Brazilian states).

For each state we calculate:
- Total orders
- Late orders (Late + Super Late)
- Late percentage
- Average review score

In [ ]:
# Create a 'is_late' column for easier calculation
# 1 = late or super late, 0 = on time

master_df['is_late'] = master_df['delivery_status'].isin(['Late', 'Super Late']).astype(int)

# Group by customer state and calculate metrics
state_analysis = master_df.groupby('customer_state').agg(
    total_orders=('order_id', 'count'),
    late_orders=('is_late', 'sum'),
    avg_review=('review_score', 'mean')
).reset_index()

# Calculate late percentage
state_analysis['late_percentage'] = (
    state_analysis['late_orders'] / state_analysis['total_orders'] * 100
).round(2)

# Sort by late percentage descending
state_analysis = state_analysis.sort_values('late_percentage', ascending=False)

print('=== STATE DELIVERY PERFORMANCE ===')
print(state_analysis.to_string(index=False))

In [ ]:
# Create chart: Top states by late percentage
# We take top 15 states for better readability

top_states = state_analysis.head(15)

fig = px.bar(
    top_states,
    x='customer_state',
    y='late_percentage',
    title='Top 15 States by Late Delivery Percentage',
    labels={'customer_state': 'State', 'late_percentage': 'Late %'},
    color='late_percentage',
    color_continuous_scale='Reds'
)
fig.update_layout(xaxis_tickangle=-45)
fig.show()

---
## Section 8: Sentiment Analysis

We analyze how delivery performance affects customer satisfaction.

**Hypothesis:** Late deliveries lead to lower review scores.

We compare:
- Average review score for On Time, Late, and Super Late deliveries
- Relationship between delay days and review score

In [ ]:
# Calculate average review score by delivery status
sentiment_analysis = master_df.groupby('delivery_status').agg(
    avg_review=('review_score', 'mean'),
    count=('order_id', 'count')
).reset_index()

# Sort by delivery severity
status_order = ['On Time', 'Late', 'Super Late']
sentiment_analysis['delivery_status'] = pd.Categorical(
    sentiment_analysis['delivery_status'],
    categories=status_order,
    ordered=True
)
sentiment_analysis = sentiment_analysis.sort_values('delivery_status')

print('=== DELIVERY STATUS vs REVIEW SCORE ===')
print(sentiment_analysis.to_string(index=False))

In [ ]:
# Chart 1: Delivery Status vs Average Review Score

fig = px.bar(
    sentiment_analysis,
    x='delivery_status',
    y='avg_review',
    title='Delivery Status vs Average Review Score',
    labels={'delivery_status': 'Delivery Status', 'avg_review': 'Average Review Score'},
    color='delivery_status',
    color_discrete_map={'On Time': 'green', 'Late': 'orange', 'Super Late': 'red'}
)
fig.update_layout(yaxis_range=[0, 5])
fig.show()

In [ ]:
# Chart 2: Delay Days vs Review Score
# We group delay_days into bins to see the trend clearly

# Create delay bins
master_df['delay_bin'] = pd.cut(
    master_df['delay_days'],
    bins=[-100, 0, 5, 10, 15, 20, 30, 100],
    labels=['Early', '0-5 days', '6-10 days', '11-15 days', '16-20 days', '21-30 days', '30+ days']
)

# Calculate average review by delay bin
delay_review = master_df.groupby('delay_bin', observed=True).agg(
    avg_review=('review_score', 'mean'),
    count=('order_id', 'count')
).reset_index()

fig = px.bar(
    delay_review,
    x='delay_bin',
    y='avg_review',
    title='Delay Days vs Average Review Score',
    labels={'delay_bin': 'Delay Period', 'avg_review': 'Average Review Score'},
    color='avg_review',
    color_continuous_scale='RdYlGn'
)
fig.update_layout(yaxis_range=[0, 5])
fig.show()

---
## Section 9: Product Category Analysis

We analyze which product categories have the most delivery problems.

Using the English translated category names, we find:
- Categories with highest late delivery percentage
- Categories with lowest average review scores

In [ ]:
# Filter out categories with too few orders (less than 100)
# This removes noise from categories with small sample sizes

category_analysis = master_df.groupby('product_category_name_english').agg(
    total_orders=('order_id', 'count'),
    late_orders=('is_late', 'sum'),
    avg_review=('review_score', 'mean'),
    avg_delay=('delay_days', 'mean')
).reset_index()

# Calculate late percentage
category_analysis['late_percentage'] = (
    category_analysis['late_orders'] / category_analysis['total_orders'] * 100
).round(2)

# Filter categories with at least 100 orders
category_analysis = category_analysis[category_analysis['total_orders'] >= 100]

# Sort by late percentage
category_analysis = category_analysis.sort_values('late_percentage', ascending=False)

print(f'Categories with 100+ orders: {len(category_analysis)}')
print('\n=== TOP 10 CATEGORIES BY LATE DELIVERY PERCENTAGE ===')
print(category_analysis.head(10).to_string(index=False))

In [ ]:
# Chart: Top 10 categories by late delivery percentage

top_categories = category_analysis.head(10)

fig = px.bar(
    top_categories,
    x='product_category_name_english',
    y='late_percentage',
    title='Top 10 Categories with Most Delivery Problems',
    labels={'product_category_name_english': 'Category', 'late_percentage': 'Late %'},
    color='late_percentage',
    color_continuous_scale='Reds'
)
fig.update_layout(xaxis_tickangle=-45)
fig.show()

In [ ]:
# Chart: Categories by average delay days

top_delay = category_analysis.sort_values('avg_delay', ascending=False).head(10)

fig = px.bar(
    top_delay,
    x='product_category_name_english',
    y='avg_delay',
    title='Top 10 Categories by Average Delay Days',
    labels={'product_category_name_english': 'Category', 'avg_delay': 'Average Delay (days)'},
    color='avg_delay',
    color_continuous_scale='Oranges'
)
fig.update_layout(xaxis_tickangle=-45)
fig.show()

---
## Section 10: Candidate Choice Feature

**State Risk Score** is a custom feature we create to identify regions where delivery problems are most damaging to customer satisfaction.

**Formula:**
```
risk_score = late_percentage * (5 - average_review_score)
```

**Why this matters:**
- A state with high late percentage AND low review scores gets a high risk score
- This helps the company prioritize which regions to improve first
- It combines both delivery performance and customer impact

In [ ]:
# Calculate State Risk Score
# risk_score = late_percentage * (5 - average_review_score)
# Higher score = more urgent need for improvement

state_analysis['risk_score'] = (
    state_analysis['late_percentage'] * (5 - state_analysis['avg_review'])
).round(2)

# Sort by risk score descending
state_analysis = state_analysis.sort_values('risk_score', ascending=False)

print('=== STATE RISK SCORES ===')
print(state_analysis[['customer_state', 'total_orders', 'late_percentage', 'avg_review', 'risk_score']].to_string(index=False))

In [ ]:
# Chart: Top 10 states by risk score

top_risk = state_analysis.head(10)

fig = px.bar(
    top_risk,
    x='customer_state',
    y='risk_score',
    title='Top 10 States by Risk Score (Late % x Customer Impact)',
    labels={'customer_state': 'State', 'risk_score': 'Risk Score'},
    color='risk_score',
    color_continuous_scale='reds'
)
fig.update_layout(xaxis_tickangle=-45)
fig.show()

In [ ]:
# Explain the risk score to interviewers

print('=== RISK SCORE EXPLANATION ===')
print('\nThe Risk Score helps identify where to focus improvement efforts.')
print('\nFormula: risk_score = late_percentage * (5 - avg_review)')
print('\nInterpretation:')
print('  - High late_percentage + Low avg_review = HIGH RISK')
print('  - This means: Many late deliveries AND customers are unhappy')
print('  - The company should prioritize logistics improvements in these states')

---
## Section 11: Export Final Dataset

We save the cleaned and enriched dataset to `outputs/cleaned_delivery_data.csv`.

This file can be used for:
- Building dashboards
- Further analysis
- Sharing with stakeholders

In [ ]:
# Create outputs directory if it doesn't exist
import os
os.makedirs('../outputs', exist_ok=True)

# Select columns to export
# We keep the most important columns for delivery analysis

export_columns = [
    'order_id',
    'customer_id',
    'customer_state',
    'customer_city',
    'order_status',
    'order_purchase_timestamp',
    'order_delivered_customer_date',
    'order_estimated_delivery_date',
    'review_score',
    'product_id',
    'product_category_name_english',
    'delay_days',
    'days_difference',
    'delivery_status',
    'is_late'
]

# Export to CSV
master_df[export_columns].to_csv('../outputs/cleaned_delivery_data.csv', index=False)

print(f'Exported {len(master_df):,} rows to outputs/cleaned_delivery_data.csv')
print(f'Columns: {len(export_columns)}')

In [ ]:
# Final summary

print('=' * 60)
print('LAST MILE LOGISTICS AUDITOR - ANALYSIS COMPLETE')
print('=' * 60)
print(f'\nTotal orders analyzed: {len(master_df):,}')
print(f'On Time deliveries: {(master_df["delivery_status"] == "On Time").sum():,}')
print(f'Late deliveries: {(master_df["delivery_status"] == "Late").sum():,}')
print(f'Super Late deliveries: {(master_df["delivery_status"] == "Super Late").sum():,}')
print(f'\nAverage review score: {master_df["review_score"].mean():.2f}')
print(f'Average delay: {master_df["delay_days"].mean():.1f} days')
print(f'\nOutput file: outputs/cleaned_delivery_data.csv')